#Task 1:Data Preprocessing cleanup

In [0]:
import pandas as pd
import numpy as np

In [0]:
#load data
spark.sql("USE `patient-readmission`.patient_readmission_schema")  # set default catalog/schema so the bare table name resolves
df = spark.table("readmission_patient").toPandas()
print(df.shape) # rows, columns
print(df.dtypes) # data type of each column 

In [0]:
display(df)

In [0]:
 df.info(), df.describe(include="all")

In [0]:
#checking for null values
df.isnull().sum()


In [0]:
#checking for duplicates
df.duplicated().sum()


In [0]:
# Task 1b: Derived clinical bucket columns (analysis-only – the split cell later excludes them from model features)
import pandas as pd

# 1. Age Bucket – life stages (dataset ages are 18-89, so there is no Children group)
# Youth 18-24, Young Adult 25-39, Middle-Aged 40-59, Elders 60+
df['Age Bucket'] = pd.cut(df['Age'], bins=[0, 25, 40, 60, 150],
                          labels=['Youth (18-24)', 'Young Adult (25-39)', 'Middle-Aged (40-59)', 'Elders (60+)'], right=False)

# 2. Blood Sugar Level (mg/dL): Low <70, Normal 70-139, High >=140
df['Blood Sugar Level'] = pd.cut(df['Blood Sugar Levels'], bins=[0, 70, 140, 1000],
                                 labels=['Low', 'Normal', 'High'], right=False)

# 3. Blood Pressure Level (systolic mmHg): Low <90, Normal 90-139, High >=140
df['Blood Pressure Level'] = pd.cut(df['Blood Pressure'], bins=[0, 90, 140, 300],
                                    labels=['Low', 'Normal', 'High'], right=False)

# 4. Length of Stay Category (days): Low <7, Medium 7-14, High 15+
df['Length of Stay Category'] = pd.cut(df['Length of Stay'], bins=[0, 7, 15, 1000],
                                       labels=['Low (<7 days)', 'Medium (7-14 days)', 'High (15+ days)'], right=False)

# Distribution of each new bucket column
for col in ['Age Bucket', 'Blood Sugar Level', 'Blood Pressure Level', 'Length of Stay Category']:
    print(f"\n{col}:")
    print(df[col].value_counts().sort_index())

# Preview original value next to its bucket
display(df[['Age', 'Age Bucket',
            'Blood Sugar Levels', 'Blood Sugar Level',
            'Blood Pressure', 'Blood Pressure Level',
            'Length of Stay', 'Length of Stay Category']].head(10))

In [0]:
df['Gender']=df['Gender'].str.strip().str.title()
df['Admission Type']=df['Admission Type'].str.strip().str.title()
display(df)


In [0]:
# One-Hot Encoding for categoricals (pandas get_dummies — pd is already imported in Cell 1)

In [0]:
 # 1. Target – Readmission (kept as a single binary label: No=0, Yes=1)
df['Readmission'] = df['Readmission'].str.strip().str.title().map({'No': 0, 'Yes': 1})

In [0]:
 # 2. Gender – one-hot encode
df = pd.get_dummies(df, columns=['Gender'], prefix='Gender', dtype=int) # adds Gender_Female / Gender_Male

In [0]:
 # 3. Admission Type – one-hot encode
df = pd.get_dummies(df, columns=['Admission Type'], prefix='AdmissionType', dtype=int) # adds AdmissionType_Elective / _Emergency / _Urgent

In [0]:
encoded_cols = [c for c in df.columns if c.startswith('Gender') or c.startswith('AdmissionType') or c == 'Readmission']
print(df[encoded_cols])
#display only encoded columns
display(df[encoded_cols])

In [0]:
from collections import Counter
print("Count:", Counter(df['Readmission']))
print(f"Percentage Yes: {df['Readmission'].mean()*100:.1f}%")

In [0]:
  %pip install seaborn

In [0]:
%restart_python

In [0]:
 # Task 2: Distribution of target
import seaborn as sns
import matplotlib.pyplot as plt
# df is already loaded and preprocessed by the Task 1 cells above
sns.countplot(x='Readmission', data=df)
plt.title("Readmission Within 30 Days - Class Balance")
plt.xlabel("0=No, 1=Yes")
plt.show()
print(df['Readmission'].value_counts())
print(df['Readmission'].value_counts(normalize=True)*100)

In [0]:
 # Task 2: Histograms of key vitals
df[['Age','Blood Pressure','Blood Sugar Levels','Length of Stay']].hist(figsize=(10,6), bins=20)
plt.suptitle("Distribution of Clinical Variables")
plt.tight_layout()
plt.show()

In [0]:
%md>
## All pairwise correlations are close to zero(between -0.03 and 0.03), showing no strong correlation between any of the features and the target


In [0]:
 # Task 2: Correlation heatmap
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
corr = df[['Age','Length of Stay','Number of Diagnoses','Previous Admissions','Blood Pressure','Blood Sugar Levels']].corr()
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.2f')
plt.title("Correlation Between Variables and Readmission")
plt.show()

In [0]:
cols = ['Age','Length of Stay','Number of Diagnoses','Previous Admissions','Blood Pressure','Blood Sugar Levels','Readmission']
corr = df[cols].corr()

import matplotlib.pyplot as plt
plt.figure(figsize=(8,6))
plt.imshow(corr, cmap='Blues')
plt.colorbar()
plt.xticks(range(len(cols)), cols, rotation=90)
plt.yticks(range(len(cols)), cols)
plt.title("Correlation Between Variables and Readmission")
for i in range(len(cols)):
    for j in range(len(cols)):
        plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=9)
plt.tight_layout()
plt.show()

##The features are not strongly related to each other,so each one adds its own separate information to the model.

#Task 3:Feature selection 

In [0]:
 # Task 3: Prepare X (features) and y (target) and Split
# Patient ID is just an identifier, not a predictive feature – drop it
# Readmission is our target – drop from X, keep as y
# Bucket columns from Task 1b are analysis-only (text categories), so drop them from model features
X = df.drop(['Patient ID', 'Readmission', 'Age Bucket', 'Blood Sugar Level', 'Blood Pressure Level', 'Length of Stay Category'], axis=1)
y = df['Readmission']

# Train/Test Split 80/20
# test_size=0.2 = 20% for testing, 80% for training
# random_state=42 = reproducibility, same split every run
# stratify=y = keeps same Yes/No ratio in both train and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("y_train distribution:", y_train.value_counts().to_dict())

## What `stratify=y` does in the train/test split

**Without stratification**, `train_test_split` shuffles all 3000 patients and picks 600 for the test set *purely at random*. With a minority class (only 28.9% Yes = 866 patients), random sampling can get unlucky - the test set might end up with 24% Yes or 33% Yes just by chance, especially with small data.

**With `stratify=y`**, the split is done *within each class separately*: scikit-learn takes the 2134 No patients and samples 20% of them (427) for the test set, then takes the 866 Yes patients and samples 20% of them (173) for the test set. Both sets are then merged.

### Why it matters for this model

* **Representative evaluation** - the test set mirrors the real 71/29 No/Yes mix, so precision and recall reflect how the model performs on patients like the real population, not a skewed sample.
* **Both classes present to train on** - without stratification, an unlucky split could leave the training set with too few Yes patients for the model to learn from meaningfully.
* **Stable metrics** - stratification removes class-ratio luck from the split, so differences between runs come from the model, not sampling noise.

### The numbers in this notebook

| | Yes (1) | No (0) | Yes % |
| --- | --- | --- | --- |
| Full data | 866 | 2134 | 28.9% |
| Train (80%) | 693 | 1707 | 28.9% |
| Test (20%) | 173 | 427 | 28.9% |

All three rows carry exactly the same rate - that is stratification working: 866 x 0.2 = 173.2 -> 173, and 2134 x 0.2 = 426.8 -> 427. Without `stratify=y`, those percentages would drift from 28.9%.

**Note:** `stratify` balances the *class ratio*, not the feature distributions - it does not guarantee equal numbers of elderly patients or emergency admissions across sets, just the No/Yes outcome mix.



In [0]:
%pip install imbalanced-learn  # provides the imblearn package (SMOTE)

In [0]:
#Task 3b: Handle Class Imbalance using SMOTE
# WHY: y_train is 1707 vs 693 (71% vs 29%). Model would be biased to class 0.
# SMOTE = Synthetic Minority Oversampling Technique
# It creates synthetic new rows for class 1 by interpolating between existing class 1 samples
# IMPORTANT: Only apply to X_train, y_train – NEVER to X_test. Test must stay real.

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_balanced.value_counts().to_dict())
# After should be {0: 1707, 1: 1707}

In [0]:
#show only patients who were readmitted
display(X_train_balanced[y_train_balanced == 1].head(10))


#Model building
##80/20 split +SMOTE+Random Forest

In [0]:
 # Task 4:Scale + Better Model
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# 1. Scale both balanced train and test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

# 2. Use RandomForest – works better with this data than Logistic Regression
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train_balanced)

# 3. Predict
y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

## Task 4 result: the baseline model misses most readmissions

Despite SMOTE balancing the training data, the Random Forest at the default 0.5 threshold catches only **8% of readmitted patients** (14 of 173). The 65% accuracy is misleading — it comes mostly from correctly predicting the majority (No) class. Precision is also low at 22%. This is the classic class-imbalance failure: good overall accuracy, poor performance on the class we actually care about.


In [0]:
# Task 4b: Improving Recall - Threshold Tuning

from sklearn.metrics import classification_report, confusion_matrix, recall_score, precision_score
from sklearn.ensemble import RandomForestClassifier

# Retrain model here so this cell works on its own
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train_balanced)

# Get probabilities, not just 0 / 1
y_proba = model.predict_proba(X_test_scaled)[:, 1]

# Try different thresholds and see recall
for threshold in [0.5, 0.4, 0.35, 0.3, 0.25, 0.2]:
    y_pred_new = (y_proba >= threshold).astype(int)
    print(f"Threshold: {threshold}")
    print(f"Recall for Yes: {recall_score(y_test, y_pred_new):.2f}")
    print(f"Precision for Yes: {precision_score(y_test, y_pred_new):.2f}")
    print("---")




## Threshold tuning: trading precision for recall

Lowering the decision threshold flags more patients as high-risk. At 0.25, recall rises from 8% to **76%** while precision holds around 29%. For readmission screening this trade is intentional: a missed readmission (false negative) is clinically costlier than an unnecessary follow-up (false positive), so we accept more false positives to catch more true cases.


In [0]:
 # Final Model - Threshold 0.25

best_threshold = 0.25
y_pred_final = (y_proba >= best_threshold).astype(int)

print("Confusion Matrix - Final Model (Threshold 0.25):")
print(confusion_matrix(y_test, y_pred_final))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_final))

##On 600 test patients:

- *109:* Correctly predicted No, and they were No. True Negatives.
- *318:* Predicted Yes, but they were actually No. False Positives. This is now very high.
- *42:* Predicted No, but they were actually Yes. False Negatives. This is now very low, which was your goal.
- *131:* Correctly predicted Yes, and they were Yes. True Positives.


In [0]:
from sklearn.ensemble import RandomForestClassifier

# No SMOTE needed here, class_weight does the balancing
model2 = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    class_weight='balanced_subsample',
    random_state=42
)

model2.fit(scaler.transform(X_train), y_train) # Use ORIGINAL X_train (2400 rows), not the SMOTE-scaled one (3414 rows)

y_proba2 = model2.predict_proba(X_test_scaled)[:, 1]
y_pred2 = (y_proba2 >= 0.35).astype(int)

print(confusion_matrix(y_test, y_pred2))
print(classification_report(y_test, y_pred2))


## Final model: 92% recall

Replacing SMOTE with `class_weight='balanced_subsample'` and using a 0.35 threshold catches **160 of 173 readmissions (92% recall)**, leaving only 13 false negatives. Precision stays at 29% — of the 551 patients flagged as high-risk, about 1 in 3 actually returns. Accuracy drops to 33%: the accepted price of a screening-first model that refuses to miss readmitted patients.


In [0]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
log_model.fit(scaler.transform(X_train), y_train) # Use ORIGINAL X_train (2400 rows), not the SMOTE-scaled one (3414 rows)

y_proba_log = log_model.predict_proba(X_test_scaled)[:, 1]

# Find best threshold for at least 75% recall
for threshold in [0.5, 0.4, 0.3, 0.25]:
    y_pred_log = (y_proba_log >= threshold).astype(int)
    print(f"Threshold {threshold} - Recall: {recall_score(y_test, y_pred_log):.2f} - Precision: {precision_score(y_test, y_pred_log):.2f}")


## Why Logistic Regression was rejected

At any threshold below 0.4 the logistic model predicts Yes for **every** patient — recall is 100% but precision is 29% and the model has zero discrimination (it cannot tell readmitted from non-readmitted patients apart at all). The Random Forest with class weights was kept as the final model.


In [0]:
%md>

### Three models were compared. The baseline Random Forest achieved only 8% recall. Threshold tuning improved recall to 76%. The final model, a Random Forest with class_weight='balanced_subsample' and a threshold of 0.35, achieved 92% recall, reducing false negatives from 159 to 13. Precision remained at 29% and accuracy dropped to 33%, showing the trade-off of prioritising recall for clinical screening. Logistic Regression was rejected because it predicted Yes for all patients at thresholds below 0.5, achieving 100% recall but providing no discrimination.


#Task 5:Model evaluation and testing

In [0]:
 # Task 5:Which Factors drive readmission? in %
import pandas as pd

importances = model.feature_importances_
feat = pd.DataFrame({'feature': X.columns, 'importance': importances})
feat['importance_%'] = feat['importance'] * 100
feat = feat.sort_values('importance_%', ascending=False)

# Display as % with 2 decimals
pd.set_option('display.float_format', lambda x: f'{x:.2f}%')
print(feat[['feature', 'importance_%']])

In [0]:
 # Task 5 with chart
import pandas as pd
import matplotlib.pyplot as plt

importances = model.feature_importances_
feat = pd.DataFrame({'feature': X.columns, 'importance': importances}).sort_values('importance', ascending=True)

plt.figure(figsize=(8,5))
plt.barh(feat['feature'], feat['importance'])
plt.xlabel("Importance")
plt.title("Which factors drive readmission?")
plt.tight_layout()
plt.show()



In [0]:
 # Task 5b: Feature Importance for Final Model - Self-contained

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

# Retrain Model 2 here so this cell works on its own
model2 = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    class_weight='balanced_subsample',
    random_state=42
)
model2.fit(scaler.transform(X_train), y_train) # Use ORIGINAL X_train (2400 rows), not the SMOTE-scaled one (3414 rows)

# Get importances
importances = model2.feature_importances_
feat = pd.DataFrame({
    'feature': X.columns,
    'importance': importances
})
feat['importance_%'] = feat['importance'] * 100
feat = feat.sort_values('importance', ascending=True)

print(feat)

plt.figure(figsize=(10, 6))
plt.barh(feat['feature'], feat['importance_%'], color='skyblue')
plt.xlabel('Importance (%)')
plt.title('Feature Importance - Final Model (Random Forest + class_weight)')
plt.tight_layout()
plt.show()

 ##For the final model (Random Forest with class_weight='balanced_subsample'), the most important predictors of readmission were Blood Sugar Levels (22.5%), Blood Pressure (19.0%), Age (17.2%), and Length of Stay (16.6%). Together, these four clinical factors accounted for over 75% of the model's decision-making. Number of Diagnoses (9.7%) and Previous Admissions (7.3%) were moderately important, while Gender and Admission Type contributed less than 3% each and were not strong predictors of readmission.

 #This suggests hospitals should prioritise close monitoring and follow-up for older patients with unstable blood sugar, high blood pressure, and longer initial stays, as they are at highest risk of readmission.


In [0]:
 # Readmission rate by age group
age_rate = (df.groupby('Age Bucket', observed=True)
              .agg(patients=('Readmission', 'size'),
                   readmitted=('Readmission', 'sum'),
                   readmission_rate=('Readmission', 'mean')))
age_rate['readmission_rate'] = (age_rate['readmission_rate'] * 100).round(1)
print(age_rate)

# Overall rate for comparison
print(f"\nOverall readmission rate: {df['Readmission'].mean()*100:.1f}%")

In [0]:
 # Visualize readmission rate by age group
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.bar(age_rate.index.astype(str), age_rate['readmission_rate'], color='steelblue', edgecolor='black')
plt.axhline(df['Readmission'].mean()*100, color='red', linestyle='--', label=f"Overall: {df['Readmission'].mean()*100:.1f}%")
plt.xlabel('Age Group')
plt.ylabel('Readmission Rate (%)')
plt.title('Readmission Rate by Age Group')
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
 # Readmission rate by blood sugar level
sugar_rate = (df.groupby('Blood Sugar Level', observed=True)
                 .agg(patients=('Readmission', 'size'),
                      readmitted=('Readmission', 'sum'),
                      readmission_rate=('Readmission', 'mean')))
sugar_rate['readmission_rate'] = (sugar_rate['readmission_rate'] * 100).round(1)
print(sugar_rate)

# Overall rate for comparison
print(f"\nOverall readmission rate: {df['Readmission'].mean()*100:.1f}%")

In [0]:
 # Visualize readmission rate by blood sugar level
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.bar(sugar_rate.index.astype(str), sugar_rate['readmission_rate'], color='seagreen', edgecolor='black')
plt.axhline(df['Readmission'].mean()*100, color='red', linestyle='--', label=f"Overall: {df['Readmission'].mean()*100:.1f}%")
plt.xlabel('Blood Sugar Level')
plt.ylabel('Readmission Rate (%)')
plt.title('Readmission Rate by Blood Sugar Level')
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
 # Readmission rate by blood pressure level
bp_rate = (df.groupby('Blood Pressure Level', observed=True)
              .agg(patients=('Readmission', 'size'),
                   readmitted=('Readmission', 'sum'),
                   readmission_rate=('Readmission', 'mean')))
bp_rate['readmission_rate'] = (bp_rate['readmission_rate'] * 100).round(1)
print(bp_rate)

# Overall rate for comparison
print(f"\nOverall readmission rate: {df['Readmission'].mean()*100:.1f}%")

In [0]:
 # Visualize readmission rate by blood pressure level
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.bar(bp_rate.index.astype(str), bp_rate['readmission_rate'], color='indianred', edgecolor='black')
plt.axhline(df['Readmission'].mean()*100, color='red', linestyle='--', label=f"Overall: {df['Readmission'].mean()*100:.1f}%")
plt.xlabel('Blood Pressure Level')
plt.ylabel('Readmission Rate (%)')
plt.title('Readmission Rate by Blood Pressure Level')
plt.legend()
plt.tight_layout()
plt.show()